# Auto Face Crop - Colab/Kaggle Deployment

Notebook untuk deploy aplikasi auto face crop di Google Colab atau Kaggle dengan port forwarding.

## Setup Steps:
1. Jalankan semua cells secara berurutan
2. Dapatkan URL publik dari output ngrok
3. Akses aplikasi via URL tersebut

## 1. Check GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install pyngrok for port forwarding
!pip install -q pyngrok

## 2. Clone Repository

⚠️ **Ganti URL repository dengan repository Anda sendiri**

In [ ]:
# Clone repository (ganti dengan URL repository Anda)
!git clone https://github.com/USERNAME/auto_crop_face.git
%cd auto_crop_face

## 3. Install Project Dependencies

In [ ]:
# Install requirements
!pip install -q -r requirements.txt

## 4. Configure Ngrok

⚠️ **Dapatkan auth token gratis dari [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)**

Ganti `YOUR_NGROK_TOKEN` dengan token Anda.

In [ ]:
from pyngrok import ngrok, conf

# Set ngrok auth token (GANTI INI!)
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # Dapatkan dari https://dashboard.ngrok.com/get-started/your-authtoken

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    print("⚠️  PERINGATAN: Silakan ganti YOUR_NGROK_TOKEN dengan token ngrok Anda!")
    print("    Dapatkan token gratis di: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok token configured!")

## 5. Start Flask App with Port Forwarding

In [ ]:
import threading
import time
from app import app
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

# Start Flask in background thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Wait for Flask to start
time.sleep(3)

# Create ngrok tunnel
try:
    public_url = ngrok.connect(5000, bind_tls=True)
    print("="*60)
    print("🚀 Auto Face Crop is running!")
    print("="*60)
    print(f"\n📱 Access the app at: {public_url}\n")
    print("="*60)
    print("\n⏱️  Keep this cell running to maintain the connection")
    print("💡 Tip: Copy the URL and open it in a new tab\n")
except Exception as e:
    print(f"❌ Error starting ngrok: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure you set NGROK_TOKEN in the previous cell")
    print("2. Check your ngrok token at https://dashboard.ngrok.com/")
    print("3. Free ngrok accounts may have rate limits")

## 6. Keep Alive

Jalankan cell ini untuk menjaga aplikasi tetap hidup. Tekan **Interrupt** (⬛) untuk menghentikan.

In [ ]:
import time

print("⏳ Keeping the app alive...")
print("   Press the Stop button (⬛) to shutdown\n")

try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\n\n🛑 Shutting down...")
    ngrok.kill()
    print("✅ Cleanup complete")

---

## Alternative: Kaggle Deployment (No Ngrok)

Jika menggunakan Kaggle, Anda bisa mengakses aplikasi via Kaggle's built-in proxy tanpa ngrok.

Uncomment dan jalankan cell di bawah:

In [ ]:
# # Kaggle Alternative (without ngrok)
# import threading
# from app import app
# 
# def run_flask():
#     app.run(host='0.0.0.0', port=5000, debug=False)
# 
# flask_thread = threading.Thread(target=run_flask, daemon=True)
# flask_thread.start()
# 
# print("🚀 Flask app is running on port 5000")
# print("📱 Access via Kaggle's output interface")
# 
# # Keep alive
# import time
# try:
#     while True:
#         time.sleep(60)
# except KeyboardInterrupt:
#     print("Shutting down...")

---

## Troubleshooting

### Ngrok Error: "authtoken not found"
- Pastikan Anda sudah set `NGROK_TOKEN` di Cell 4
- Dapatkan token gratis dari https://dashboard.ngrok.com/

### MediaPipe Installation Error
```python
!pip install --upgrade pip
!pip install mediapipe --no-cache-dir
```

### Out of Memory (OOM)
- Pastikan runtime menggunakan GPU (Runtime → Change runtime type → GPU)
- Untuk video besar, aplikasi otomatis limit ke 300 frames

### Port Already in Use
- Restart runtime: Runtime → Restart runtime
- Atau kill ngrok: `ngrok.kill()`